In [1]:
import pandas as pd
import sqlite3

# 1. Chargement des 4 fichiers CSV
print("Chargement des fichiers en mémoire...")
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')

# 2. Création de la base de données locale
conn = sqlite3.connect('ecommerce.db')

# 3. Transfert des données vers des tables SQL
print("Création des tables SQL...")
df_orders.to_sql('orders', conn, if_exists='replace', index=False)
df_items.to_sql('order_items', conn, if_exists='replace', index=False)
df_customers.to_sql('customers', conn, if_exists='replace', index=False)
df_products.to_sql('products', conn, if_exists='replace', index=False)

# 4. LA REQUÊTE SQL (Le cœur de votre compétence Data Analyst)
# Nous lions les commandes aux clients, puis aux articles, puis aux produits.
requete_master = """
SELECT
    o.order_id,
    o.order_purchase_timestamp AS date_achat,
    o.order_status AS statut_commande,
    c.customer_city AS ville_client,
    c.customer_state AS etat_client,
    i.price AS prix_produit,
    i.freight_value AS frais_port,
    p.product_category_name AS categorie_produit
FROM
    orders o
JOIN
    customers c ON o.customer_id = c.customer_id
JOIN
    order_items i ON o.order_id = i.order_id
JOIN
    products p ON i.product_id = p.product_id
WHERE
    o.order_status = 'delivered' -- Pertinence métier : on ne garde que les commandes finalisées
"""

# 5. Exécution de la requête
print("Exécution de la jointure SQL...")
df_master = pd.read_sql_query(requete_master, conn)

# On affiche un aperçu pour vérifier
print("\nDimensions de la table finale :", df_master.shape)
display(df_master.head())

# 6. Exportation pour la Data Visualization
df_master.to_csv('master_ecommerce.csv', index=False)
print("\n Fichier 'master_ecommerce.csv' généré avec succès ! Vous pouvez le télécharger.")

Chargement des fichiers en mémoire...
Création des tables SQL...
Exécution de la jointure SQL...

Dimensions de la table finale : (110197, 8)


,order_id,date_achat,statut_commande,ville_client,etat_client,prix_produit,frais_port,categorie_produit
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,delivered,sao paulo,SP,29.99,8.72,utilidades_domesticas
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,delivered,barreiras,BA,118.70,22.76,perfumaria
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,delivered,vianopolis,GO,159.90,19.22,automotivo
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,delivered,sao goncalo do amarante,RN,45.00,27.20,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,delivered,santo andre,SP,19.90,8.72,papelaria



 Fichier 'master_ecommerce.csv' généré avec succès ! Vous pouvez le télécharger.
